# The Kickoff Report | NFL game predictions
*Otto Ganser*

Built The Kickoff Report to estimate NFL win probabilities and project both teams' scores before kickoff. I started with recent scoring and rest, then tested whether team efficiency, Elo ratings, and projected quarterback performance improved the forecasts. The final section exports upcoming 2026 matchups for the website.

**Data sources** · [nflverse schedules and results](https://github.com/nflverse/nfldata/blob/master/data/games.csv) · [Schedule column definitions](https://nflreadr.nflverse.com/articles/dictionary_schedules.html)

**Reproducibility** · The notebook runs top to bottom in Jupyter or VS Code and uses pandas, scikit-learn, pyarrow, and certifi.

## 1. Schedule and results

I use the nflverse schedule as the source for matchups, kickoff information, and final scores. The notebook prefers a local copy named `games.csv` when available and otherwise reads the [raw nflverse schedule](https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv). Setting `USE_LOCAL_IF_AVAILABLE = False` makes nflverse the direct source on each run.

In [ ]:
from pathlib import Path
from urllib.error import URLError

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_URL = "https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv"
LOCAL_CSV = Path("games.csv")
USE_LOCAL_IF_AVAILABLE = True
START_SEASON = 2015
VALIDATION_SEASON = 2024
TEST_SEASON = 2025
PREDICTION_SEASON = 2026
LOOKBACK = 5

columns = [
    "game_id", "season", "game_type", "week", "gameday", "gametime",
    "location", "home_team", "away_team", "home_score", "away_score",
]
source = LOCAL_CSV if USE_LOCAL_IF_AVAILABLE and LOCAL_CSV.exists() else DATA_URL

try:
    schedule = pd.read_csv(source, usecols=columns, parse_dates=["gameday"])
except URLError as error:
    raise RuntimeError(
        f"Could not download the schedule. Save games.csv in {Path.cwd()} "
        "and set USE_LOCAL_IF_AVAILABLE = True, then rerun this cell."
    ) from error

schedule = (
    schedule.loc[schedule["season"].ge(START_SEASON - 1)]
    .sort_values(["gameday", "gametime", "game_id"])
    .reset_index(drop=True)
)
assert schedule["game_id"].is_unique, "Expected one row per game."

# The schedule lists kickoff times in Eastern time, including international games.
kickoff = pd.to_datetime(
    schedule["gameday"].dt.strftime("%Y-%m-%d") + " " + schedule["gametime"],
    format="%Y-%m-%d %H:%M",
    errors="coerce",
)
schedule["kickoff_et"] = kickoff.dt.tz_localize("America/New_York")
schedule["has_result"] = schedule[["home_score", "away_score"]].notna().all(axis=1)
schedule["point_margin"] = schedule["home_score"] - schedule["away_score"]

print(f"Loaded {len(schedule):,} games from {source}")
print(f"Working folder: {Path.cwd()}")
display(
    schedule.groupby("season").agg(
        scheduled_games=("game_id", "size"), completed_games=("has_result", "sum")
    ).tail(6)
)

## 2. Team game history

I reshape completed games into two records, one from each team's perspective. Playoff results contribute to team history, while the matchups I train on, evaluate, and forecast are regular-season games.

For each matchup, I only use team results from earlier calendar dates. This keeps the game's own outcome out of its inputs and applies the same rule in historical evaluation and forecasting. The five-game window can carry over between seasons, and I cap rest at 21 days so the offseason does not dominate that feature.

In [ ]:
# Treat franchise moves and abbreviation changes as continuous team histories.
franchise_aliases = {
    "OAK": "LV", "STL": "LA", "LAR": "LA", "SD": "LAC", "WSH": "WAS",
}

completed = schedule.loc[schedule["has_result"]].copy()
home_history = completed[["game_id", "gameday", "home_team", "home_score", "away_score"]].copy()
home_history.columns = ["game_id", "gameday", "team", "points_for", "points_against"]
away_history = completed[["game_id", "gameday", "away_team", "away_score", "home_score"]].copy()
away_history.columns = home_history.columns

team_history = pd.concat([home_history, away_history], ignore_index=True)
team_history["team"] = team_history["team"].replace(franchise_aliases)
team_history = team_history.sort_values("gameday")
history_by_team = dict(tuple(team_history.groupby("team")))

display(team_history.tail())

## 3. Scoring and rest baseline

My baseline uses both teams' average points scored, average points allowed, and days since their last game. I also flag whether the designated home team is playing at home, which accounts for neutral-site games. The 2014 schedule gives early 2015 matchups some prior history.

In [ ]:
feature_cols = [
    "home_points_for_avg", "home_points_against_avg", "home_rest_days",
    "away_points_for_avg", "away_points_against_avg", "away_rest_days",
    "home_field",
]

def team_form(team, game_date):
    team = franchise_aliases.get(team, team)
    history = history_by_team.get(team)
    if history is None or pd.isna(game_date):
        return np.nan, np.nan, np.nan

    prior_games = history.loc[history["gameday"].lt(game_date)].tail(LOOKBACK)
    if len(prior_games) < LOOKBACK:
        return np.nan, np.nan, np.nan

    days_since_last_game = (game_date - prior_games["gameday"].iloc[-1]).days
    return (
        prior_games["points_for"].mean(),
        prior_games["points_against"].mean(),
        min(days_since_last_game, 21),
    )

def add_pregame_features(matchups):
    rows = []
    for game in matchups.itertuples(index=False):
        home_form = team_form(game.home_team, game.gameday)
        away_form = team_form(game.away_team, game.gameday)
        home_field = {"Home": 1, "Neutral": 0}.get(game.location, np.nan)
        rows.append((*home_form, *away_form, home_field))

    features = pd.DataFrame(rows, columns=feature_cols, index=matchups.index)
    return matchups.join(features)

historical_games = completed.loc[
    completed["season"].between(START_SEASON, TEST_SEASON)
    & completed["game_type"].eq("REG")
]
history_features = add_pregame_features(historical_games)
model_data = history_features.dropna(subset=feature_cols).copy()
model_data["home_win"] = model_data["point_margin"].gt(0).astype(int)

print(f"Model-ready games: {len(model_data):,} of {len(historical_games):,}")
display(model_data[["game_id", *feature_cols]].head())

## 4. Offensive and defensive efficiency

I tested expected points added (EPA) per play as a measure of how efficiently each team performed. EPA measures how much a play changes the offense's expected points. I use run and pass plays with EPA from nflverse, including sacks classified as passes. Across each team's previous five completed games, I divide offensive EPA by offensive plays and opponent EPA by defensive plays. A higher EPA allowed means worse defensive performance.

The first run caches one Parquet file per season, roughly 250 MB in total. Historical files are reused, and the current-season file is refreshed when its cache is at least 24 hours old. The [nflverse play-by-play releases](https://github.com/nflverse/nflverse-data/releases/tag/pbp) provide the same files for a manual cache setup in `pbp_cache`.

In [ ]:
import shutil
import ssl
import subprocess
import time
from urllib.request import urlopen

import certifi

PBP_URL = "https://github.com/nflverse/nflverse-data/releases/download/pbp"
PBP_CACHE = Path("pbp_cache")
PBP_CACHE.mkdir(exist_ok=True)
ssl_context = ssl.create_default_context(cafile=certifi.where())
pbp_columns = [
    "game_id", "posteam", "defteam", "play_type", "epa",
    "qb_dropback", "qb_scramble", "passer_player_id", "rusher_player_id",
]

def load_pbp_season(season):
    path = PBP_CACHE / f"play_by_play_{season}.parquet"
    exists = path.exists()
    refresh = season == PREDICTION_SEASON and exists and time.time() - path.stat().st_mtime >= 86400

    if not exists or refresh:
        url = f"{PBP_URL}/{path.name}"
        partial = path.with_suffix(".part")
        print(f"Downloading {path.name}...", flush=True)
        try:
            # curl uses the system certificate setup on macOS.
            if shutil.which("curl"):
                subprocess.run(
                    ["curl", "-fLsS", "--retry", "1", "--max-time", "90",
                     url, "-o", str(partial)],
                    check=True,
                )
            else:
                with urlopen(url, context=ssl_context, timeout=60) as response:
                    with partial.open("wb") as output:
                        shutil.copyfileobj(response, output)
            partial.replace(path)
        except (OSError, subprocess.CalledProcessError) as error:
            partial.unlink(missing_ok=True)
            if exists:
                print(f"Refresh failed. Using cached {path.name}.")
            elif season == PREDICTION_SEASON:
                print(f"Could not download {path.name}. Some forecasts may use the baseline.")
                return None
            else:
                raise RuntimeError(f"Download {url} and save it to {path.resolve()}, then rerun.") from error

    return pd.read_parquet(path, columns=pbp_columns)

def summarize_epa(plays):
    plays = plays.loc[plays["play_type"].isin(["pass", "run"])].dropna(
        subset=["game_id", "posteam", "defteam", "epa"]
    )
    offense = (
        plays.groupby(["game_id", "posteam"], observed=True)["epa"]
        .agg(off_epa_sum="sum", off_plays="size")
        .rename_axis(index=["game_id", "team"])
    )
    defense = (
        plays.groupby(["game_id", "defteam"], observed=True)["epa"]
        .agg(def_epa_allowed_sum="sum", def_plays="size")
        .rename_axis(index=["game_id", "team"])
    )
    return offense.join(defense, how="outer").reset_index()

def summarize_qb_epa(plays):
    dropbacks = plays.loc[plays["qb_dropback"].eq(1)].copy()
    dropbacks["qb_id"] = dropbacks["passer_player_id"].where(
        ~dropbacks["qb_scramble"].eq(1), dropbacks["rusher_player_id"]
    )
    dropbacks = dropbacks.dropna(subset=["game_id", "posteam", "qb_id", "epa"])
    return (
        dropbacks.groupby(["game_id", "posteam", "qb_id"], as_index=False, observed=True)
        .agg(total_epa=("epa", "sum"), dropbacks=("epa", "size"))
    )

epa_by_season = []
qb_by_season = []
for season in range(START_SEASON - 1, PREDICTION_SEASON + 1):
    plays = load_pbp_season(season)
    if plays is not None:
        epa_by_season.append(summarize_epa(plays))
        qb_by_season.append(summarize_qb_epa(plays))

team_game_epa = pd.concat(epa_by_season, ignore_index=True)
team_game_epa["team"] = team_game_epa["team"].replace(franchise_aliases)
epa_history = team_history.merge(
    team_game_epa, on=["game_id", "team"], how="left", validate="one_to_one"
).sort_values("gameday")
epa_history_by_team = dict(tuple(epa_history.groupby("team")))

print(f"Matched EPA to {epa_history['off_plays'].notna().sum():,} completed team-game records.")

In [ ]:
epa_cols = [
    "home_off_epa_per_play", "home_def_epa_allowed_per_play",
    "away_off_epa_per_play", "away_def_epa_allowed_per_play",
]
epa_feature_cols = feature_cols + epa_cols
epa_totals = ["off_epa_sum", "off_plays", "def_epa_allowed_sum", "def_plays"]

def team_epa_form(team, game_date):
    team = franchise_aliases.get(team, team)
    history = epa_history_by_team.get(team)
    if history is None or pd.isna(game_date):
        return np.nan, np.nan

    prior = history.loc[history["gameday"].lt(game_date)].tail(LOOKBACK)
    totals = prior[epa_totals].sum(min_count=LOOKBACK)
    return (
        totals["off_epa_sum"] / totals["off_plays"],
        totals["def_epa_allowed_sum"] / totals["def_plays"],
    )

def add_epa_features(matchups):
    rows = [
        (*team_epa_form(game.home_team, game.gameday),
         *team_epa_form(game.away_team, game.gameday))
        for game in matchups.itertuples(index=False)
    ]
    epa_features = pd.DataFrame(rows, columns=epa_cols, index=matchups.index)
    return matchups.join(epa_features)

epa_model_data = add_epa_features(model_data).dropna(subset=epa_cols)
print(f"EPA-ready historical games: {len(epa_model_data):,} of {len(model_data):,}")
display(epa_model_data[["game_id", *epa_cols]].head())

## 5a. Pregame Elo ratings

I added Elo to capture team strength in the context of the opponents each team has played. I initialize each team at 1,500 in 2014 and use a 20-point update factor, a 55-point home-field adjustment, and 25% regression toward 1,500 between seasons.

The model records the home-minus-away Elo gap, including home-field advantage where applicable, before processing that date's results. Playoff games update team ratings, and ties count as half a win.

In [ ]:
ELO_START = 1500
ELO_K = 20
ELO_HOME_ADVANTAGE = 55
ELO_OFFSEASON_RETAIN = 0.75
elo_cols = ["pregame_elo_gap"]

def build_elo_features(games):
    ratings = {}
    previous_season = None
    gaps = {}

    for (season, _), day in games.groupby(["season", "gameday"], sort=True):
        if previous_season is not None and season != previous_season:
            ratings = {
                team: ELO_START + ELO_OFFSEASON_RETAIN * (rating - ELO_START)
                for team, rating in ratings.items()
            }
        previous_season = season
        updates = {}

        for game in day.itertuples(index=False):
            home = franchise_aliases.get(game.home_team, game.home_team)
            away = franchise_aliases.get(game.away_team, game.away_team)
            home_rating = ratings.get(home, ELO_START)
            away_rating = ratings.get(away, ELO_START)
            advantage = ELO_HOME_ADVANTAGE if game.location == "Home" else 0
            gap = home_rating - away_rating + advantage
            gaps[game.game_id] = gap

            if game.has_result:
                expected_home = 1 / (1 + 10 ** (-gap / 400))
                actual_home = (np.sign(game.point_margin) + 1) / 2
                change = ELO_K * (actual_home - expected_home)
                updates[home] = updates.get(home, 0) + change
                updates[away] = updates.get(away, 0) - change

        for team, change in updates.items():
            ratings[team] = ratings.get(team, ELO_START) + change

    return pd.Series(gaps, name=elo_cols[0], dtype=float)

pregame_elo = build_elo_features(schedule)

def add_elo_features(matchups):
    return matchups.join(pregame_elo, on="game_id")

elo_model_data = add_elo_features(model_data).dropna(subset=elo_cols)
comparison_data = add_elo_features(epa_model_data).dropna(subset=elo_cols)
print(f"Games available for model comparison: {len(comparison_data):,}")
display(elo_model_data[["game_id", *elo_cols]].head())

## 5b. Quarterback efficiency and starter changes

I tested whether accounting for the projected starting quarterback adds information beyond team-level performance. I use nflverse play-by-play through 2026 and assign passes and sacks to the passer and scrambles to the rusher. A QB's rating is their total EPA across the last five eligible appearances divided by their total dropbacks plus 50. The extra 50 dropbacks pull small samples toward zero EPA per dropback.

The historical forecast cutoff is Tuesday at 9:15 a.m. Central each week, or kickoff if earlier. For upcoming games I use the notebook run time. Both use QB appearances from earlier calendar dates.

**QB Change experiment.** I also tested the difference between the projected starter's rating and the team's combined QB EPA per dropback over its previous five completed games. I apply the same 50-dropback adjustment to both ratings and subtract the away team's difference from the home team's difference. A team's adjustment is zero if either rating lacks prior dropbacks.

### Projected starters and availability

I start from the most recent timestamped depth chart available by the cutoff. For upcoming matchups, I skip QBs listed as out or on reserve for that week. Questionable and doubtful starters remain in the main forecast, with a comparison against the next available QB when listed.

I use the previous game's primary QB as a fallback when depth-chart data is unavailable. If all listed QBs are unavailable, the model uses a neutral rating. Because the weekly injury and roster files lack publication timestamps, I use them only for upcoming games. Historical evaluation uses timestamped depth charts and the previous-game fallback. The 2025 projected-starter comparison is limited to matchups with depth charts for both teams at the cutoff.

In [ ]:
LINEUP_CACHE = Path("lineup_cache")
LINEUP_CACHE.mkdir(exist_ok=True)
LINEUP_BASE = "https://github.com/nflverse/nflverse-data/releases/download"

def load_lineup_csv(tag, name, columns, refresh):
    path = LINEUP_CACHE / name
    stale = path.exists() and time.time() - path.stat().st_mtime >= 3600
    if not path.exists() or (refresh and stale):
        url = f"{LINEUP_BASE}/{tag}/{name}"
        partial = path.with_suffix(".part")
        try:
            if shutil.which("curl"):
                subprocess.run(["curl", "-fLsS", "--max-time", "30", url, "-o", str(partial)], check=True)
            else:
                with urlopen(url, context=ssl_context, timeout=30) as response, partial.open("wb") as output:
                    shutil.copyfileobj(response, output)
            partial.replace(path)
        except (OSError, subprocess.CalledProcessError):
            partial.unlink(missing_ok=True)
            print(f"Could not refresh {name}; using available local data.")
    if not path.exists():
        return pd.DataFrame(columns=columns)
    return pd.read_csv(path, usecols=columns, dtype={"gsis_id": "string"})

depth_cols = ["dt", "team", "gsis_id", "player_name", "pos_abb", "pos_rank"]
depth = pd.concat([
    load_lineup_csv("depth_charts", f"depth_charts_{season}.csv", depth_cols, season == PREDICTION_SEASON)
    for season in range(2025, PREDICTION_SEASON + 1)
], ignore_index=True)
depth = depth.loc[depth["pos_abb"].eq("QB")].dropna(subset=["gsis_id", "pos_rank"]).copy()
depth["dt"] = pd.to_datetime(depth["dt"], utc=True, errors="coerce")
depth["team"] = depth["team"].replace(franchise_aliases)
depth = depth.dropna(subset=["dt"]).sort_values(["dt", "pos_rank", "gsis_id"])
depth_by_team = dict(tuple(depth.groupby("team")))

injury_cols = ["season", "week", "team", "gsis_id", "position", "report_status", "game_type"]
injuries = load_lineup_csv("injuries", f"injuries_{PREDICTION_SEASON}.csv", injury_cols, True)
injuries = injuries.loc[injuries["position"].eq("QB") & injuries["game_type"].eq("REG")].copy()
injuries["team"] = injuries["team"].replace(franchise_aliases)

roster_cols = ["season", "week", "team", "gsis_id", "position", "status", "game_type"]
roster = load_lineup_csv("weekly_rosters", f"roster_weekly_{PREDICTION_SEASON}.csv", roster_cols, True)
roster = roster.loc[roster["position"].eq("QB") & roster["game_type"].eq("REG")].copy()
roster["team"] = roster["team"].replace(franchise_aliases)

availability = {}
for row in injuries.dropna(subset=["season", "week", "gsis_id", "report_status"]).itertuples():
    key = (int(row.season), int(row.week), row.team, row.gsis_id)
    status = str(row.report_status).strip().lower()
    if status in {"questionable", "doubtful", "out"}:
        order = {"questionable": 1, "doubtful": 2, "out": 3}
        prior = availability.get(key)
        if order[status] > order.get(prior, 0):
            availability[key] = status

for row in roster.loc[roster["status"].eq("RES")].dropna(subset=["season", "week", "gsis_id"]).itertuples():
    availability[(int(row.season), int(row.week), row.team, row.gsis_id)] = "reserve"

print(f"Loaded {len(depth):,} timestamped QB depth-chart rows and {len(availability):,} availability flags.")

In [ ]:
QB_PRIOR_DROPBACKS = 50
qb_feature_cols = ["qb_epa_gap", "qb_epa_mean"]
qb_change_cols = ["qb_change_gap"]

qb_history = pd.concat(qb_by_season, ignore_index=True).merge(
    completed[["game_id", "gameday"]], on="game_id", how="inner", validate="many_to_one"
)
qb_history["team"] = qb_history["posteam"].replace(franchise_aliases)
qb_history = qb_history.sort_values(["gameday", "game_id"])
qb_history_by_player = dict(tuple(qb_history.groupby("qb_id")))
team_qb_games = (
    qb_history.groupby(["team", "game_id", "gameday"], as_index=False)
    .agg(total_epa=("total_epa", "sum"), dropbacks=("dropbacks", "sum"))
    .sort_values(["gameday", "game_id"])
)
team_qb_history = dict(tuple(team_qb_games.groupby("team")))

# Pick the player with the most dropbacks in each team's completed game.
primary_qbs = (
    qb_history.sort_values(["game_id", "team", "dropbacks", "qb_id"], ascending=[True, True, False, True])
    .drop_duplicates(["game_id", "team"])
    .sort_values(["gameday", "game_id"])
)
primary_qbs_by_team = dict(tuple(primary_qbs.groupby("team")))

regular_schedule = schedule.loc[schedule["game_type"].eq("REG")].copy()
first_kickoff = regular_schedule.groupby(["season", "week"])["kickoff_et"].transform("min")
week_start = first_kickoff.dt.tz_convert("America/Chicago").dt.tz_localize(None).dt.normalize()
tuesday = week_start - pd.to_timedelta((week_start.dt.dayofweek - 1) % 7, unit="D")
weekly_cutoff = (tuesday + pd.Timedelta(hours=9, minutes=15)).dt.tz_localize("America/Chicago")
kickoff_ct = regular_schedule["kickoff_et"].dt.tz_convert("America/Chicago")
historical_cutoff = dict(zip(
    regular_schedule["game_id"],
    weekly_cutoff.where(weekly_cutoff.lt(kickoff_ct), kickoff_ct),
))

def expected_qb(team, game, cutoff, date, use_depth, live):
    team = franchise_aliases.get(team, team)

    def status(qb_id):
        if not live:
            return ""
        return availability.get((game.season, game.week, team, qb_id), "")
    unavailable = {"out", "reserve"}
    if use_depth and pd.notna(cutoff):
        depth_history = depth_by_team.get(team)
        if depth_history is not None:
            known = depth_history.loc[depth_history["dt"].le(cutoff)]
            if not known.empty and cutoff - known["dt"].iloc[-1] <= pd.Timedelta(days=14):
                snapshot = known.loc[known["dt"].eq(known["dt"].iloc[-1])]
                options = list(dict.fromkeys(snapshot["gsis_id"]))
                eligible = [qb_id for qb_id in options if status(qb_id) not in unavailable]
                if not eligible:
                    return None, "depth chart: all unavailable", "", None
                starter = eligible[0]
                starter_status = status(starter)
                alternate = eligible[1] if len(eligible) > 1 and starter_status in {"questionable", "doubtful"} else None
                source = "depth chart" if starter == options[0] else "depth chart + availability"
                return starter, source, starter_status, alternate

    history = primary_qbs_by_team.get(team)
    if history is not None:
        known = history.loc[history["gameday"].lt(date)]
        if not known.empty:
            previous = known["qb_id"].iloc[-1]
            if status(previous) not in unavailable:
                return previous, "previous game", status(previous), None
    return None, "unknown", "", None

def qb_rating(qb_id, date):
    history = qb_history_by_player.get(qb_id)
    if history is None:
        return 0.0, 0
    prior = history.loc[history["gameday"].lt(date)].tail(LOOKBACK)
    attempts = int(prior["dropbacks"].sum())
    return prior["total_epa"].sum() / (attempts + QB_PRIOR_DROPBACKS), attempts

def team_qb_reference(team, date):
    history = team_qb_history.get(franchise_aliases.get(team, team))
    if history is None:
        return 0.0, 0
    prior = history.loc[history["gameday"].lt(date)].tail(LOOKBACK)
    attempts = int(prior["dropbacks"].sum())
    return prior["total_epa"].sum() / (attempts + QB_PRIOR_DROPBACKS), attempts

def qb_change(starter_rating, starter_dropbacks, team_rating, team_dropbacks):
    if not starter_dropbacks or not team_dropbacks:
        return 0.0
    return starter_rating - team_rating

def changed_starter(team, qb_id, date):
    history = primary_qbs_by_team.get(franchise_aliases.get(team, team))
    if history is None or qb_id is None:
        return False
    prior = history.loc[history["gameday"].lt(date)]
    return not prior.empty and qb_id != prior["qb_id"].iloc[-1]

def add_qb_features(matchups, as_of=None, use_depth=True):
    rows = []
    for game in matchups.itertuples(index=False):
        cutoff = historical_cutoff.get(game.game_id, pd.NaT) if as_of is None else as_of
        date = cutoff.tz_convert("America/Chicago").tz_localize(None).normalize() if pd.notna(cutoff) else pd.NaT
        live = as_of is not None
        home_id, home_source, home_status, home_alt = expected_qb(
            game.home_team, game, cutoff, date, use_depth, live
        )
        away_id, away_source, away_status, away_alt = expected_qb(
            game.away_team, game, cutoff, date, use_depth, live
        )
        home, home_n = qb_rating(home_id, date)
        away, away_n = qb_rating(away_id, date)
        home_ref, home_ref_n = team_qb_reference(game.home_team, date)
        away_ref, away_ref_n = team_qb_reference(game.away_team, date)
        home_change = qb_change(home, home_n, home_ref, home_ref_n)
        away_change = qb_change(away, away_n, away_ref, away_ref_n)
        starter_changed = (
            changed_starter(game.home_team, home_id, date)
            or changed_starter(game.away_team, away_id, date)
        )
        rows.append((cutoff, home_id, away_id, home_source, away_source, home, away,
                     home_n, away_n, home_status, away_status, home_alt, away_alt,
                     home - away, (home + away) / 2,
                     home_ref, away_ref, home_ref_n, away_ref_n,
                     home_change, away_change, starter_changed, home_change - away_change))
    columns = [
        "qb_cutoff", "home_expected_qb", "away_expected_qb", "home_qb_source", "away_qb_source",
        "home_qb_epa", "away_qb_epa", "home_qb_dropbacks", "away_qb_dropbacks",
        "home_qb_status", "away_qb_status", "home_alternate_qb", "away_alternate_qb",
        *qb_feature_cols,
        "home_qb_reference", "away_qb_reference", "home_qb_reference_dropbacks", "away_qb_reference_dropbacks",
        "home_qb_change", "away_qb_change", "projected_starter_change", *qb_change_cols,
    ]
    return matchups.join(pd.DataFrame(rows, columns=columns, index=matchups.index))

qb_model_data = add_qb_features(elo_model_data)
print(f"Loaded {len(qb_history):,} completed QB appearances.")
print(f"Matchups with both QB ratings based on prior dropbacks: "
      f"{(qb_model_data[['home_qb_dropbacks', 'away_qb_dropbacks']] > 0).all(axis=1).sum():,}")
display(qb_model_data.loc[
    qb_model_data["season"].eq(2024) & qb_model_data["week"].eq(6),
    ["game_id", "qb_cutoff", "home_expected_qb", "away_expected_qb",
     "home_qb_epa", "away_qb_epa", "home_qb_dropbacks", "away_qb_dropbacks"],
].head())

## 6. Model evaluation and selection

I compare the scoring-and-rest baseline, team EPA, Elo alone, and combinations that add Elo or QB features to the baseline. I evaluate the models on the same eligible games for each comparison. For each validation season from 2022 through 2024, I train on earlier seasons. The 2025 reference trains through 2023 to match the original benchmark. Earlier results within an evaluation season can still update features for later matchups.

I use logistic regression for winners and ridge regression for scores. Winner models exclude ties, while score models include them. I report winner accuracy, log loss for win probabilities, and mean absolute error for scores.

The default export uses my original rule for the two Elo variants. A variant must beat the scoring-and-rest baseline in at least two of the three validation seasons and improve average accuracy. If both qualify, I choose the higher average accuracy and use lower log loss to break a tie. I use the previously reviewed 2025 results as a reference and evaluate the QB variants separately.

In the 2026 replay, I fit through 2025 and compare Baseline + Elo with the QB Change variant on the same completed games. I also report the subset where the projected starter differs from the previous game's primary QB. The replay excludes untimestamped live availability reports.

In [ ]:
score_cols = ["home_score", "away_score"]

def fit_models(games, inputs):
    decisive_games = games.loc[games["point_margin"].ne(0)]
    winner = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    winner.fit(decisive_games[inputs], decisive_games["home_win"])

    scores = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    scores.fit(games[inputs], games[score_cols])
    return winner, scores

def projected_points(model, features):
    return model.predict(features).clip(min=0)

def evaluate(games, winner, scores, inputs):
    decisive = games.loc[games["point_margin"].ne(0)]
    actual_winners = decisive["home_win"]
    home_prob = winner.predict_proba(decisive[inputs])[:, 1]
    predicted_scores = projected_points(scores, games[inputs])

    return {
        "Winner games": len(decisive),
        "Always-home accuracy": actual_winners.mean(),
        "Winner accuracy": accuracy_score(actual_winners, home_prob >= 0.5),
        "Log loss": log_loss(actual_winners, home_prob, labels=[0, 1]),
        "Score MAE": mean_absolute_error(games[score_cols], predicted_scores),
    }


model_features = {
    "Baseline": feature_cols,
    "Baseline + EPA": epa_feature_cols,
    "Elo only": elo_cols,
    "Baseline + Elo": feature_cols + elo_cols,
    "Baseline + Elo + QB": feature_cols + elo_cols + qb_feature_cols,
    "Baseline + Elo + QB Change": feature_cols + elo_cols + qb_change_cols,
}
selection_seasons = list(range(VALIDATION_SEASON - 2, VALIDATION_SEASON + 1))

paired_data = comparison_data.join(qb_model_data[
    qb_feature_cols + qb_change_cols + ["projected_starter_change"]
])
results = {}
starter_change_results = {}
for season in [*selection_seasons, TEST_SEASON]:
    train_before = min(season, VALIDATION_SEASON)
    train = paired_data.loc[paired_data["season"].lt(train_before)]
    held_out = paired_data.loc[paired_data["season"].eq(season)]
    if train.empty or held_out.empty:
        raise ValueError(f"Missing comparable data for the {season} evaluation.")

    for name, inputs in model_features.items():
        winner, scores = fit_models(train, inputs)
        results[(season, name)] = evaluate(held_out, winner, scores, inputs)
        changed = held_out.loc[held_out["projected_starter_change"]]
        if name in {"Baseline + Elo", "Baseline + Elo + QB Change"} and changed["point_margin"].ne(0).any():
            starter_change_results[(season, name)] = evaluate(changed, winner, scores, inputs)

metrics = pd.DataFrame.from_dict(results, orient="index")
metrics.index.names = ["Season", "Model"]
metrics["Winner games"] = metrics["Winner games"].astype(int)
display(metrics.round(3))

validation = metrics.loc[selection_seasons]
accuracy_by_season = validation["Winner accuracy"].unstack("Model")
validation_summary = validation.groupby(level="Model")[["Winner accuracy", "Log loss", "Score MAE"]].mean()
validation_summary["Seasons beating baseline"] = accuracy_by_season.gt(
    accuracy_by_season["Baseline"], axis="index"
).sum()
display(validation_summary.round(3))

qb_comparison = metrics.xs("Baseline + Elo + QB", level="Model")[
    ["Winner accuracy", "Log loss", "Score MAE"]
] - metrics.xs("Baseline + Elo", level="Model")[["Winner accuracy", "Log loss", "Score MAE"]]
print("QB model minus Baseline + Elo on the same games:")
display(qb_comparison.round(4))

if starter_change_results:
    starter_metrics = pd.DataFrame.from_dict(starter_change_results, orient="index")
    starter_metrics.index.names = ["Season", "Model"]
    print("Games with at least one projected QB different from the team's previous-game primary QB:")
    display(starter_metrics.round(4))

elo_options = validation_summary.loc[["Elo only", "Baseline + Elo"]]
qualified = elo_options.loc[
    elo_options["Seasons beating baseline"].ge(2)
    & elo_options["Winner accuracy"].gt(validation_summary.loc["Baseline", "Winner accuracy"])
].sort_values(["Winner accuracy", "Log loss"], ascending=[False, True])
selected_model = "Baseline" if qualified.empty else qualified.index[0]
print(f"Selected website model: {selected_model}")

# Compare starter selection rules using identical QB-model weights and games.
projected = qb_model_data.loc[
    qb_model_data["season"].eq(TEST_SEASON)
    & qb_model_data["home_qb_source"].eq("depth chart")
    & qb_model_data["away_qb_source"].eq("depth chart")
]
if not projected.empty and projected["home_win"].nunique() == 2:
    inputs = model_features["Baseline + Elo + QB"]
    fit_data = qb_model_data.loc[qb_model_data["season"].lt(VALIDATION_SEASON)]
    qb_winner, qb_scores = fit_models(fit_data, inputs)
    previous = add_qb_features(elo_model_data.loc[projected.index], use_depth=False)
    print("2025 starter selection comparison on the same depth-chart-covered games:")
    display(pd.DataFrame({
        "Previous-game QB": evaluate(previous, qb_winner, qb_scores, inputs),
        "Timestamped depth chart": evaluate(projected, qb_winner, qb_scores, inputs),
    }).T.round(4))

# A newer evaluation uses only models fitted through 2025 and completed 2026 games.
recent = completed.loc[
    completed["season"].eq(PREDICTION_SEASON) & completed["game_type"].eq("REG")
]
recent = add_qb_features(add_elo_features(add_pregame_features(recent)))
recent = recent.dropna(subset=feature_cols + elo_cols + qb_change_cols).copy()
recent["home_win"] = recent["point_margin"].gt(0).astype(int)
recent_results = {}
for name in ("Baseline + Elo", "Baseline + Elo + QB Change"):
    inputs = model_features[name]
    winner, scores = fit_models(qb_model_data, inputs)
    for label, sample in [
        ("All completed games", recent),
        ("Projected starter change", recent.loc[recent["projected_starter_change"]]),
    ]:
        if sample["point_margin"].ne(0).any():
            recent_results[(label, name)] = evaluate(sample, winner, scores, inputs)
if recent_results:
    recent_metrics = pd.DataFrame.from_dict(recent_results, orient="index")
    recent_metrics.index.names = ["Sample", "Model"]
    print(f"{PREDICTION_SEASON} replay using timestamped depth charts and training through {TEST_SEASON}:")
    display(recent_metrics.round(4))


## 7. Upcoming 2026 forecasts

I refit the selected winner and score models through 2025 and generate forecasts using the latest eligible results and pregame features. As the season progresses, new results update those features. If a matchup lacks inputs for the selected model, it can fall back to the scoring-and-rest baseline. The `model_used` column records the model that produced each forecast.

The winner probability and score projection are fitted separately, so they can favor different teams. I use `PREDICTION_MODEL` to compare alternative forecasts, including `Baseline + Elo + QB` and `Baseline + Elo + QB Change`. The QB diagnostic table records starter IDs, lineup sources, and prior dropbacks.

In [ ]:
# Set to "Baseline + Elo + QB" or "Baseline + Elo + QB Change" to try a QB model.
PREDICTION_MODEL = selected_model
as_of = pd.Timestamp.now(tz="America/New_York")
upcoming = schedule.loc[
    schedule["season"].eq(PREDICTION_SEASON)
    & schedule["game_type"].eq("REG")
    & ~schedule["has_result"]
    & schedule["kickoff_et"].gt(as_of)
]
ready = add_qb_features(
    add_elo_features(add_epa_features(add_pregame_features(upcoming))), as_of=as_of
)
qb_names = depth.dropna(subset=["player_name"]).drop_duplicates("gsis_id", keep="last").set_index("gsis_id")["player_name"]
for side in ("home", "away"):
    ids = ready[f"{side}_expected_qb"]
    ready[f"{side}_qb_name"] = ids.map(qb_names).fillna(ids).fillna("Unknown")

qb_preview_cols = [
    "game_id", "away_qb_name", "away_qb_source", "away_qb_status",
    "home_qb_name", "home_qb_source", "home_qb_status",
    "home_qb_epa", "away_qb_epa", "home_qb_change", "away_qb_change",
]
if not ready.empty:
    display(ready.loc[ready["week"].eq(ready["week"].min()), qb_preview_cols])

output_cols = ["game_id", "season", "week", "kickoff_et", "away_team", "home_team"]

def predict_matchups(matchups, training_games, inputs, model_name):
    output = matchups[output_cols].copy()
    if output.empty:
        return output

    winner, scores = fit_models(training_games, inputs)
    features = matchups[inputs]
    home_prob = winner.predict_proba(features)[:, 1]
    points = projected_points(scores, features)
    output["home_win_probability"] = home_prob
    output["away_win_probability"] = 1 - home_prob
    output["model_pick"] = np.where(home_prob >= 0.5, matchups["home_team"], matchups["away_team"])
    output["projected_home_score"] = np.rint(points[:, 0]).astype(int)
    output["projected_away_score"] = np.rint(points[:, 1]).astype(int)
    output["prediction_time_et"] = as_of
    output["model_used"] = model_name
    return output

training_data = {
    "Baseline": model_data,
    "Baseline + EPA": epa_model_data,
    "Elo only": elo_model_data,
    "Baseline + Elo": elo_model_data,
    "Baseline + Elo + QB": qb_model_data,
    "Baseline + Elo + QB Change": qb_model_data,
}
chosen_inputs = model_features[PREDICTION_MODEL]
has_chosen_inputs = ready[chosen_inputs].notna().all(axis=1)
has_baseline_inputs = ready[feature_cols].notna().all(axis=1)
predictions = pd.concat([
    predict_matchups(
        ready.loc[has_chosen_inputs], training_data[PREDICTION_MODEL], chosen_inputs, PREDICTION_MODEL
    ),
    predict_matchups(
        ready.loc[~has_chosen_inputs & has_baseline_inputs], model_data, feature_cols, "Baseline"
    ),
]).sort_values("kickoff_et")

if PREDICTION_MODEL in {"Baseline + Elo + QB", "Baseline + Elo + QB Change"} and not ready.empty:
    alternatives = []
    cutoff_date = as_of.tz_convert("America/Chicago").tz_localize(None).normalize()
    for side in ("home", "away"):
        alternate_col = f"{side}_alternate_qb"
        scenario = ready.loc[ready[alternate_col].notna()].copy()
        if scenario.empty:
            continue
        alternate_rating = scenario[alternate_col].map(lambda qb_id: qb_rating(qb_id, cutoff_date))
        scenario[f"{side}_qb_epa"] = alternate_rating.map(lambda rating: rating[0])
        scenario[f"{side}_qb_change"] = [
            qb_change(rating, attempts, ref, ref_attempts)
            for (rating, attempts), ref, ref_attempts in zip(
                alternate_rating, scenario[f"{side}_qb_reference"],
                scenario[f"{side}_qb_reference_dropbacks"]
            )
        ]
        scenario["qb_change_gap"] = scenario["home_qb_change"] - scenario["away_qb_change"]
        scenario["qb_epa_gap"] = scenario["home_qb_epa"] - scenario["away_qb_epa"]
        scenario["qb_epa_mean"] = (scenario["home_qb_epa"] + scenario["away_qb_epa"]) / 2
        alternate = predict_matchups(scenario, qb_model_data, chosen_inputs, PREDICTION_MODEL)
        alternate["alternate_qb"] = scenario[alternate_col].map(qb_names).fillna(scenario[alternate_col])
        alternate["changed_side"] = side
        alternate["original_home_win_probability"] = predictions["home_win_probability"].reindex(alternate.index)
        alternatives.append(alternate)
    if alternatives:
        print("Backup scenarios for questionable or doubtful QBs:")
        display(pd.concat(alternatives)[[
            "game_id", "changed_side", "alternate_qb", "original_home_win_probability",
            "home_win_probability", "projected_home_score", "projected_away_score",
        ]].round(3))

if not predictions.empty:
    projected_score_cols = ["projected_home_score", "projected_away_score"]
    predictions[projected_score_cols] = predictions[projected_score_cols].astype(int)
    next_week = predictions["week"].min()
    next_games = predictions.loc[predictions["week"].eq(next_week)]
    print(f"Showing Week {next_week}. Scored {len(predictions)} future games in total.")
    print(predictions["model_used"].value_counts().to_string())
    display(next_games.round({"home_win_probability": 3, "away_win_probability": 3}))
elif upcoming.empty:
    print(f"No future regular-season kickoffs found for {PREDICTION_SEASON}. Refresh the schedule if needed.")

if len(predictions) < len(upcoming):
    print(f"Skipped {len(upcoming) - len(predictions)} matchups with incomplete features.")

## 8. Website export

I export the upcoming matchup forecasts with win probabilities, projected scores, and the model used for each game. A dated export saved before kickoff lets me match the original forecasts to final results by `game_id`.

In [ ]:
if not predictions.empty:
    output_path = Path(f"upcoming_nfl_predictions_{PREDICTION_SEASON}.csv")
    predictions.to_csv(output_path, index=False)
    print(f"Saved {len(predictions)} predictions to {output_path.resolve()}")
else:
    print("No predictions to export.")